# 04 · Regresión logística y scorecard de puntos

**Objetivo:** estimar la PD de cada cliente con una regresión logística sobre las variables WoE y convertir el modelo en un **scorecard**: una tabla de puntos fácil de leer, aplicar y explicar.

## Conceptos

### 1. Regresión logística

Estima la probabilidad de ser malo (la **PD**) a partir de las variables:

$$\text{PD} = \frac{1}{1 + e^{-(\beta_0 + \beta_1 \cdot \text{WoE}_1 + \dots + \beta_k \cdot \text{WoE}_k)}}$$

- Cada **coeficiente** (β) dice cuánto pesa la variable en el riesgo.
- Como el WoE es **positivo en los tramos de menor riesgo**, todos los coeficientes deben salir **negativos**: a mayor WoE, menor PD. Un coeficiente positivo contradice la lógica del WoE y suele indicar que dos variables se están "peleando" la misma información (multicolinealidad); esa variable se elimina.
- El **p-valor** dice si el efecto de la variable es real o podría ser casualidad. Se exige un p-valor menor a 0.05.

Es el modelo estándar para scorecards porque es **transparente**: se puede explicar a la dirección, a auditoría y al regulador exactamente por qué un cliente recibió su calificación.

### 2. De probabilidades a puntos

Los **momios** (*odds*) son buenos por cada malo. Una PD de 2 % equivale a momios de 49 a 1. El scorecard traduce los momios a puntos con dos parámetros:

| Parámetro | Valor | Significado |
|---|---|---|
| Puntaje base | 600 | Un cliente con 600 puntos tiene momios de 50 buenos por cada malo. |
| PDO (*Points to Double the Odds*) | 20 | Cada 20 puntos adicionales, los momios se duplican: 620 puntos = 100 a 1; 580 puntos = 25 a 1. |

Así, **más puntos = menos riesgo**, igual que el score de Buró. Los puntos de cada tramo se calculan para que, al sumarlos, den exactamente el puntaje que corresponde a la PD del modelo.

## Entorno

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "credit-risk-scorecard"
CARPETA_DRIVE = Path("/content/drive/MyDrive/credit-risk-scorecard")  # Donde está cs-training.csv

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q
    os.chdir(ruta_repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import config
from src.datos import copiar_datos_crudos
from src.limpieza import obtener_datos_limpios
from src.scorecard import (
    FACTOR,
    MOMIOS_BASE,
    OFFSET,
    PDO,
    PUNTAJE_BASE,
    ajustar_logistica,
    desglose_cliente,
    graficar_distribucion_puntajes,
    pd_desde_puntaje,
    puntaje_clientes,
    tabla_calibracion,
    tabla_puntos,
)
from src.woe import preparar_woe

config.fijar_semilla()
config.crear_carpetas()
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_rows", 100)

In [ ]:
# Datos limpios y transformación WoE de las etapas 2 y 3 (se regeneran en una sesión nueva)
copiar_datos_crudos(CARPETA_DRIVE)
train, test = obtener_datos_limpios()
agrupador, seleccion, train_woe, test_woe = preparar_woe()

Y = config.COLUMNA_OBJETIVO
print(f"Variables candidatas de la etapa 3 ({len(seleccion)}):", seleccion)

## 1. Regresión logística

Se ajusta con la muestra de entrenamiento. Si alguna variable sale con signo positivo o con p-valor mayor a 0.05, se elimina y se vuelve a ajustar, de una en una.

In [ ]:
modelo, variables, eliminadas = ajustar_logistica(train_woe, seleccion)

print(f"Variables en el modelo final ({len(variables)}):", variables)
print("\nVariables eliminadas:")
display(eliminadas if not eliminadas.empty else "Ninguna")

In [ ]:
# Coeficientes del modelo final
modelo.tabla_coeficientes()

**Cómo leer la tabla:**
- `coeficiente`: todos los de las variables deben ser negativos.
- `error_estandar` y `z`: precisión del coeficiente; z = coeficiente ÷ error estándar.
- `p_valor`: probabilidad de ver un efecto así por casualidad. Menor a 0.05 = efecto real.
- El **intercepto** refleja el riesgo base de la cartera.

## 2. Scorecard: puntos por tramo

In [ ]:
print(f"Escala: {PUNTAJE_BASE} puntos = momios de {MOMIOS_BASE} a 1 | PDO = {PDO}")
print(f"factor = PDO / ln(2) = {FACTOR:.4f}")
print(f"offset = {PUNTAJE_BASE} - factor × ln({MOMIOS_BASE}) = {OFFSET:.4f}")

puntos = tabla_puntos(modelo, agrupador)
puntos

**Cómo leer el scorecard:** para calificar a un cliente se busca en qué tramo cae en cada variable y se suman los puntos. Los tramos de menor riesgo dan más puntos. El tramo "Faltante" de una variable sin faltantes en entrenamiento recibe los puntos de riesgo promedio (WoE = 0).

In [ ]:
# Rango de puntos por variable: cuánto puede mover cada variable el puntaje total
rango_puntos = (puntos.groupby(level="variable")["puntos"]
                .agg(minimo="min", maximo="max")
                .assign(rango=lambda t: t["maximo"] - t["minimo"])
                .sort_values("rango", ascending=False))
rango_puntos

## 3. Puntaje de cada cliente

In [ ]:
puntaje_train = puntaje_clientes(train, agrupador, puntos)
puntaje_test = puntaje_clientes(test, agrupador, puntos)   # Mismo scorecard, clientes nuevos

pd.DataFrame({
    "entrenamiento": puntaje_train["puntaje"].describe(),
    "prueba": puntaje_test["puntaje"].describe(),
})

In [ ]:
graficar_distribucion_puntajes(puntaje_train["puntaje"], train[Y])
plt.show()

Si el scorecard funciona, los **malos se concentran en puntajes bajos** y los **buenos en puntajes altos**. Cuanto menos se traslapen las dos distribuciones, mejor separa el modelo. En la etapa 6 esto se mide formalmente con AUC, Gini y KS.

## 4. Calibración: ¿la PD del scorecard coincide con la realidad?

Se divide a los clientes en 10 grupos por puntaje y se compara la **PD que dice el scorecard** con la **tasa de malos real** de cada grupo. Que coincidan importa porque la PD se usará en la etapa 7 para calcular la pérdida esperada.

In [ ]:
print("Entrenamiento")
display(tabla_calibracion(puntaje_train["puntaje"], train[Y]))
print("Prueba")
display(tabla_calibracion(puntaje_test["puntaje"], test[Y]))

## 5. Ejemplo: explicar la calificación de un cliente

En crédito no basta con decir "rechazado": hay que poder explicar por qué. El desglose muestra los puntos de cada variable; las de menos puntos son las **razones principales** de un puntaje bajo (*reason codes*). Se toman de la muestra de prueba el cliente con el puntaje más bajo y el de puntaje más alto.

In [ ]:
for etiqueta, id_cliente in [("más bajo", puntaje_test["puntaje"].idxmin()),
                             ("más alto", puntaje_test["puntaje"].idxmax())]:
    total = puntaje_test.loc[id_cliente, "puntaje"]
    print(f"\nCliente {id_cliente} (puntaje {etiqueta}): {total} puntos | "
          f"PD = {pd_desde_puntaje(total):.2%} | ¿Fue malo? {'Sí' if test.loc[id_cliente, Y] else 'No'}")
    display(desglose_cliente(id_cliente, test, agrupador, puntos).sort_values("puntos"))

## Resumen de la etapa

- Se ajustó una regresión logística sobre las variables WoE, con revisión de signos y p-valores.
- El modelo se convirtió en un scorecard de puntos (600 puntos = momios de 50 a 1; PDO = 20).
- Se calificó a cada cliente de entrenamiento y prueba, se revisó la calibración de la PD y se mostró cómo explicar una calificación.
- `construir_scorecard()` en `src/scorecard.py` reconstruye todo de forma determinista para los siguientes notebooks.

**Siguiente paso:** notebook `05_modelo_comparacion.ipynb`, un modelo de gradient boosting para comparar contra el scorecard.